Prueba

In [1]:
pip install hyperopt

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 5)
Dimensiones de Y: (52381, 1)


In [15]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [16]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52381, 60)


Se dividen nuevamente los conjuntos de datos

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 60)
Las dimensiones de testX son:  (10529, 60)
Las dimensiones de valX son:  (5186, 60)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

287/287 - 6s - 21ms/step - ia: 0.2695 - loss: 1.3131 - mae: 0.9393 - rmse: 1.1445 - smape: 1.4759 - val_ia: 0.2146 - val_loss: 0.8557 - val_mae: 0.7699 - val_rmse: 0.9181 - val_smape: 1.9436

Epoch 2/128                                           

287/287 - 1s - 3ms/step - ia: 0.2498 - loss: 1.1812 - mae: 0.8950 - rmse: 1.0853 - smape: 1.5046 - val_ia: 0.2163 - val_loss: 0.8508 - val_mae: 0.7675 - val_rmse: 0.9153 - val_smape: 1.8973

Epoch 3/128                                           

287/287 - 1s - 3ms/step - ia: 0.2338 - loss: 1.1057 - mae: 0.8684 - rmse: 1.0502 - smape: 1.5246 - val_ia: 0.2189 - val_loss: 0.8464 - val_mae: 0.7655 - val_rmse: 0.9127 - val_smape: 1.8175

Epoch 4/128                                           

287/287 - 1s - 4ms/step - ia: 0.2342 - loss: 1.0335 - mae: 0.8389 - rmse: 1.0150 - smape: 1.5189 - val_ia: 0.2276 - val_loss: 0.8396 - val_mae: 0.7621 - val_rmse: 0.9084 - val_smape: 1.7080

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                    

2292/2292 - 12s - 5ms/step - ia: 0.5964 - loss: 0.4923 - mae: 0.5571 - rmse: 0.6907 - smape: 0.9663 - val_ia: 0.1556 - val_loss: 1.7512 - val_mae: 1.1261 - val_rmse: 1.1706 - val_smape: 1.3821

Epoch 2/128                                                                    

2292/2292 - 7s - 3ms/step - ia: 0.6021 - loss: 0.4786 - mae: 0.5487 - rmse: 0.6816 - smape: 0.9563 - val_ia: 0.1626 - val_loss: 1.5625 - val_mae: 1.0358 - val_rmse: 1.0791 - val_smape: 1.3407

Epoch 3/128                                                                    

2292/2292 - 7s - 3ms/step - ia: 0.6018 - loss: 0.4758 - mae: 0.5474 - rmse: 0.6793 - smape: 0.9541 - val_ia: 0.1743 - val_loss: 1.3304 - val_mae: 0.9519 - val_rmse: 0.9945 - val_smape: 1.2854

Epoch 4/128                                                                    

2292/2292 - 10s - 4ms/step - ia: 0.6039 - loss: 0.4736 - mae: 0.5458 - rmse: 0.6774 - smape: 0.9

...

Epoch 1/128                                                                      

2292/2292 - 28s - 12ms/step - ia: 0.4542 - loss: 0.8855 - mae: 0.7554 - rmse: 0.9217 - smape: 1.2100 - val_ia: 0.1763 - val_loss: 1.2293 - val_mae: 0.9361 - val_rmse: 0.9731 - val_smape: 1.3459

Epoch 2/128                                                                      

2292/2292 - 8s - 3ms/step - ia: 0.5558 - loss: 0.5910 - mae: 0.6153 - rmse: 0.7572 - smape: 1.0383 - val_ia: 0.1737 - val_loss: 1.2797 - val_mae: 0.9523 - val_rmse: 0.9901 - val_smape: 1.3576

Epoch 3/128                                                                      

2292/2292 - 9s - 4ms/step - ia: 0.5869 - loss: 0.5149 - mae: 0.5707 - rmse: 0.7066 - smape: 0.9837 - val_ia: 0.1733 - val_loss: 1.2643 - val_mae: 0.9393 - val_rmse: 0.9758 - val_smape: 1.3619

Epoch 4/128                                                                      

2292/2292 - 9s - 4ms/step - ia: 0.6052 - loss: 0.4758 - mae: 0.5471 - rmse: 0.6790 - sm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

4584/4584 - 21s - 5ms/step - ia: 0.2430 - loss: 2.4719 - mae: 1.2101 - rmse: 1.4994 - smape: 1.6221 - val_ia: 0.1148 - val_loss: 2.0896 - val_mae: 1.2062 - val_rmse: 1.2217 - val_smape: 1.4669

Epoch 2/128                                                                       

4584/4584 - 14s - 3ms/step - ia: 0.2511 - loss: 2.1199 - mae: 1.1362 - rmse: 1.3955 - smape: 1.6106 - val_ia: 0.1200 - val_loss: 1.8349 - val_mae: 1.1266 - val_rmse: 1.1426 - val_smape: 1.4572

Epoch 3/128                                                                       

4584/4584 - 26s - 6ms/step - ia: 0.2549 - loss: 1.8949 - mae: 1.0820 - rmse: 1.3219 - smape: 1.6103 - val_ia: 0.1270 - val_loss: 1.6406 - val_mae: 1.0592 - val_rmse: 1.0758 - val_smape: 1.4462

Epoch 4/128                                                                       

4584/4584 - 13s - 3ms/step - ia: 0.2625 - loss: 1.7119 - mae: 1.0338 - rmse: 1.259

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

287/287 - 7s - 25ms/step - ia: 0.5369 - loss: 0.6602 - mae: 0.6523 - rmse: 0.8087 - smape: 1.0922 - val_ia: 0.4091 - val_loss: 1.4695 - val_mae: 1.0284 - val_rmse: 1.1653 - val_smape: 1.3730

Epoch 2/128                                                                         

287/287 - 1s - 4ms/step - ia: 0.5904 - loss: 0.5703 - mae: 0.6018 - rmse: 0.7539 - smape: 1.0068 - val_ia: 0.4135 - val_loss: 1.4611 - val_mae: 1.0190 - val_rmse: 1.1575 - val_smape: 1.3623

Epoch 3/128                                                                         

287/287 - 1s - 4ms/step - ia: 0.6011 - loss: 0.5365 - mae: 0.5851 - rmse: 0.7310 - smape: 0.9917 - val_ia: 0.4109 - val_loss: 1.4665 - val_mae: 1.0257 - val_rmse: 1.1583 - val_smape: 1.3731

Epoch 4/128                                                                         

287/287 - 1s - 4ms/step - ia: 0.6105 - loss: 0.5062 - mae: 0.5689 - rmse: 0.7100 -

In [23]:
print(best)

{'activation': 0, 'batch': 0, 'dropout': 0.2, 'layers': 3.0, 'learning_rate': 0.002091021650264381, 'units': 2}
